# Nigeria Flood Frequency Analysis — Anambra / Enugu Basin

Identifies chronically flooded areas east of the Niger River using **NASA OPERA DSWx-HLS**
(Dynamic Surface Water eXtent — Harmonized Landsat Sentinel) composited across five
flood seasons (Oct–Nov 2020–2024), overlaid on SRTM relief to locate upstream
intervention candidates for community-scale water catchment swales.

| | |
|---|---|
| **AOI** | Anambra / Enugu lowlands, 4.25–9.30 °E, 4.88–7.73 °N |
| **Reference event** | 2022-11-01 Nigeria floods (worst in decades) |
| **Product** | `OPERA_3_DSWX-HLS_V1` — 30 m resolution |
| **Intervention rationale** | Contour-aligned swales sequester runoff upstream of Niger tributaries; low-cost, community-scalable |

## Setup
```bash
pip install earthaccess rasterio numpy scipy geopandas shapely leafmap elevation
```
Register a free account at <https://urs.earthdata.nasa.gov/> and add credentials to `~/.netrc`:
```
machine urs.earthdata.nasa.gov login <user> password <pass>
```
Or let `earthaccess.login()` prompt you interactively.

In [1]:
!pip install earthaccess rasterio numpy scipy geopandas shapely leafmap elevation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 11.9 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 52.8 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [earthaccess]m━━━ 22/24 [s3fs]rio]

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, warnings
from pathlib import Path
import json
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.features import shapes as rio_shapes
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import box, shape, mapping
from scipy.ndimage import binary_dilation
import leafmap

sys.path.insert(0, str(Path("..").resolve()))
warnings.filterwarnings("ignore")

In [3]:
# ── AOI (parsed from the NASA Worldview URL, 2022-11-01 Nigeria floods)
WEST, SOUTH, EAST, NORTH = 4.25, 4.88, 9.30, 7.73
BBOX   = (WEST, SOUTH, EAST, NORTH)
CENTER = [(SOUTH + NORTH) / 2, (WEST + EAST) / 2]   # [lat, lon] for leafmap

# Flood season window searched per year
YEARS        = list(range(2020, 2025))   # 5 seasons → max flood frequency = 5
SEASON_START = "-10-01"
SEASON_END   = "-11-30"

# Output grid: ~30 m in degrees (DSWx-HLS native resolution)
RESOLUTION = 0.00027   # degrees per pixel
COLS = int(round((EAST  - WEST)  / RESOLUTION))
ROWS = int(round((NORTH - SOUTH) / RESOLUTION))
FREQ_TRANSFORM = from_bounds(WEST, SOUTH, EAST, NORTH, COLS, ROWS)

DATA_DIR = Path("../data/opera_dswx")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"AOI          : {WEST}°E – {EAST}°E,  {SOUTH}°N – {NORTH}°N")
print(f"Output grid  : {COLS} × {ROWS} px  ({COLS*RESOLUTION*111:.0f} × {ROWS*RESOLUTION*111:.0f} km)")
print(f"Data dir     : {DATA_DIR.resolve()}")

AOI          : 4.25°E – 9.3°E,  4.88°N – 7.73°N
Output grid  : 18704 × 10556 px  (561 × 316 km)
Data dir     : /Users/korede/code/surulere/akb/data/opera_dswx


## 1 · Data acquisition — OPERA DSWx-HLS

DSWx-HLS classifies every 30 m pixel into:

| B01_WTR value | Meaning |
|---|---|
| 0 | Not water |
| 1 | **Open water** ← counted as flooded |
| 2 | **Partial / transitional surface water** ← counted as flooded |
| 252 | Snow / Ice |
| 253 | Cloud / Cloud shadow → masked |
| 254 | Ocean masked |
| 255 | Fill / no data → masked |

For each flood season (Oct–Nov per year) we download all B01_WTR tiles covering
the AOI, mosaic them, and build a binary flood mask.

In [ ]:
import earthaccess
auth = earthaccess.login()   # reads ~/.netrc or prompts
print("Authenticated:", auth.authenticated)

In [ ]:
def fetch_dswx_year(year: int) -> list[Path]:
    """Search and download DSWx-HLS B01_WTR tiles for Oct-Nov of *year*."""
    start, end = f"{year}{SEASON_START}", f"{year}{SEASON_END}"
    year_dir = DATA_DIR / str(year)
    year_dir.mkdir(exist_ok=True)

    # Return cached files if already downloaded
    cached = sorted(year_dir.glob("*B01_WTR*.tif"))
    if cached:
        print(f"{year}: {len(cached)} cached tiles found")
        return cached

    print(f"{year}: searching {start} → {end} …", end=" ", flush=True)
    results = earthaccess.search_data(
        short_name="OPERA_L3_DSWX-HLS_V1",
        temporal=(start, end),
        bounding_box=BBOX,
        count=500,
    )
    print(f"{len(results)} granules")
    if not results:
        return []

    earthaccess.download(results, local_path=str(year_dir))
    wtr_files = sorted(year_dir.glob("*B01_WTR*.tif"))
    print(f"  → {len(wtr_files)} B01_WTR tiles")
    return wtr_files

In [ ]:
year_files: dict[int, list[Path]] = {}
for y in YEARS:
    year_files[y] = fetch_dswx_year(y)

total = sum(len(v) for v in year_files.values())
print(f"\nTotal tiles: {total}")

## 2 · Flood frequency composite

For each year:
1. Mosaic all B01_WTR tiles (taking the maximum observed water class — most conservative).
2. Threshold: pixels classified as **1 (open water) or 2 (partial water)** → flooded.
3. Reproject the binary mask onto a common WGS84 30 m grid.

Then sum the 5 annual masks: **frequency 0–5** = "flooded in N of the last 5 flood seasons".

In [ ]:
def build_water_mask(tif_files: list[Path]) -> np.ndarray | None:
    """
    Mosaic B01_WTR tiles → reproject to the shared WGS84 grid.
    Returns int8 array: 1=flooded, 0=dry, -1=cloud/nodata.
    """
    if not tif_files:
        return None

    # Mosaic: take pixel-wise maximum (open water > partial > dry)
    datasets = [rasterio.open(p) for p in tif_files]
    mosaic, mosaic_tf = merge(datasets, method="max")
    src_crs = datasets[0].crs
    for ds in datasets:
        ds.close()

    raw = mosaic[0]  # (H, W)

    # Classify
    classified = np.where(np.isin(raw, [1, 2]),    1,   # flooded
                 np.where(np.isin(raw, [252, 253, 254, 255]), -1,  # masked
                 0))                                      # dry
    classified = classified.astype(np.int8)

    # Reproject to shared WGS84 output grid
    dst = np.full((ROWS, COLS), fill_value=-1, dtype=np.int8)
    reproject(
        source=classified,
        destination=dst,
        src_transform=mosaic_tf,
        src_crs=src_crs,
        dst_transform=FREQ_TRANSFORM,
        dst_crs="EPSG:4326",
        resampling=Resampling.nearest,
    )
    return dst

In [ ]:
annual_masks: list[np.ndarray] = []

for y in YEARS:
    print(f"Processing {y} …", end=" ", flush=True)
    mask = build_water_mask(year_files[y])
    if mask is not None:
        annual_masks.append(mask)
        water_px = np.sum(mask == 1)
        print(f"{water_px:,} flooded pixels")
    else:
        print("no data")

N_YEARS = len(annual_masks)

# Stack → frequency
# Cloud/nodata (-1) treated as 0 (not flooded) — conservative
stack = np.stack([(m == 1).astype(np.uint8) for m in annual_masks], axis=0)
freq  = stack.sum(axis=0).astype(np.uint8)   # 0 … N_YEARS

print(f"\nFlood frequency map ({N_YEARS} seasons, Oct–Nov):")
labels = ["never", "1 yr", "2 yrs", "3 yrs", "4 yrs", "5 yrs"]
for v in range(N_YEARS + 1):
    n   = int(np.sum(freq == v))
    pct = 100 * n / freq.size
    print(f"  {labels[v]:>6}: {n:>9,} px  {pct:5.1f}%  {'█' * int(pct / 2)}")

In [ ]:
# Save flood frequency raster for leafmap
freq_path = DATA_DIR / "flood_frequency.tif"
with rasterio.open(
    freq_path, "w",
    driver="GTiff",
    dtype="uint8",
    width=COLS, height=ROWS, count=1,
    crs="EPSG:4326",
    transform=FREQ_TRANSFORM,
    compress="lzw",
    nodata=255,
) as dst:
    dst.write(freq, 1)

print(f"Saved → {freq_path}")

## 3 · SRTM elevation

We pull SRTM 90 m (SRTM3) for the AOI using the `elevation` library,
then resample to the same grid as the flood frequency raster.
This lets us reason about which flooded pixels sit in topographic lows
and where upstream swale sites could retain water before it reaches them.

In [ ]:
import elevation

srtm_raw  = DATA_DIR / "srtm_nigeria_90m.tif"
srtm_path = DATA_DIR / "srtm_nigeria_30m.tif"   # resampled to freq grid

if not srtm_raw.exists():
    print("Downloading SRTM 90 m …")
    elevation.clip(
        bounds=(WEST - 0.1, SOUTH - 0.1, EAST + 0.1, NORTH + 0.1),
        output=str(srtm_raw),
        product="SRTM3",
    )
    elevation.clean()
    print("Done.")
else:
    print(f"Using cached: {srtm_raw.name}")

# Resample SRTM to the freq composite grid
with rasterio.open(srtm_raw) as src:
    elev_data = np.zeros((ROWS, COLS), dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1),
        destination=elev_data,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=FREQ_TRANSFORM,
        dst_crs="EPSG:4326",
        resampling=Resampling.bilinear,
    )
    elev_nodata = src.nodata

elev_data = np.where(elev_data == elev_nodata, np.nan, elev_data)

with rasterio.open(
    srtm_path, "w",
    driver="GTiff", dtype="float32",
    width=COLS, height=ROWS, count=1,
    crs="EPSG:4326", transform=FREQ_TRANSFORM,
    compress="lzw",
) as dst:
    dst.write(elev_data, 1)

print(f"Elevation range: {np.nanmin(elev_data):.0f} m – {np.nanmax(elev_data):.0f} m")
print(f"Grid: {COLS} × {ROWS} pixels")

## 4 · Interactive flood + relief map

Layers (matching the reference NASA Worldview URL):

1. **SRTM Color Index** via NASA GIBS WMTS — same layer as in Worldview
2. **Flood frequency** (0–5 seasons, Blues ramp) — from our OPERA composite
3. **Frequently flooded** (≥ 3 of 5 seasons) — highlighted in red
4. **AOI boundary**

In [ ]:
m = leafmap.Map(center=CENTER, zoom=8, height="700px")
m.add_basemap("CartoDB.DarkMatter")

# NASA GIBS — SRTM colour-relief (same as Worldview SRTM_Color_Index layer)
m.add_tile_layer(
    url=(
        "https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/"
        "SRTM_Color_Index/default/GoogleMapsCompatible_Level8/{z}/{y}/{x}.png"
    ),
    name="SRTM Color Index (GIBS)",
    attribution="NASA GIBS / SRTM",
    opacity=0.55,
)

# Flood frequency raster
if freq_path.exists():
    m.add_raster(
        str(freq_path),
        layer_name="Flood frequency 2020–2024 (Oct–Nov)",
        colormap="Blues",
        vmin=1, vmax=5,
        opacity=0.80,
        nodata=0,   # hide never-flooded pixels
    )

# Frequently flooded overlay (≥ 3 seasons) as vector
if 'freq' in dir():
    persistent_mask = (freq >= 3).astype(np.uint8)
    geoms = [shape(g) for g, v in rio_shapes(persistent_mask, transform=FREQ_TRANSFORM) if v == 1]
    if geoms:
        gdf_persist = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")
        m.add_gdf(
            gdf_persist,
            layer_name="Frequently flooded (≥3 seasons)",
            style={"color": "#e74c3c", "fillColor": "#e74c3c",
                   "weight": 0.5, "fillOpacity": 0.45},
        )

# AOI box
aoi_gdf = gpd.GeoDataFrame(geometry=[box(*BBOX)], crs="EPSG:4326")
m.add_gdf(aoi_gdf, layer_name="AOI",
          style={"color": "#f1c40f", "fillOpacity": 0, "weight": 2})

m.add_colorbar(
    colors=["#eff3ff", "#bdd7e7", "#6baed6", "#2171b5", "#084594"],
    vmin=1, vmax=5,
    label="Flood seasons (out of 5, Oct–Nov 2020–2024)",
)
m

## 5 · Intervention site analysis

**Swale / retention bund siting criteria** (simplified topographic heuristic):

| Criterion | Value |
|---|---|
| Not in persistent flood zone | flood frequency < 3 |
| Proximate to flood zone | within ~10 km |
| Elevation above flood plain | +5 m to +50 m above flood-zone p75 |
| Below highland threshold | < 200 m (avoid steep terrain) |

In practice, field surveys would refine these using slope, land-use, and
drainage-channel mapping. This analysis surfaces plausible candidate *zones*
for community consultation, not precise engineering sites.

In [ ]:
if 'freq' in dir() and 'elev_data' in dir():
    flood_mask = freq >= 3

    # Approximate 10 km buffer in pixels (10000 m / (30 m/px) ≈ 333 px)
    near_flood = binary_dilation(flood_mask, iterations=333)

    # Elevation of the flood zone
    flood_elevs = elev_data[flood_mask & np.isfinite(elev_data)]
    if len(flood_elevs):
        floor_elev = float(np.percentile(flood_elevs, 75))
    else:
        floor_elev = 20.0
    print(f"Flood-zone elevation baseline (p75): {floor_elev:.1f} m")

    # Candidate pixels
    with np.errstate(invalid="ignore"):
        candidate = (
            near_flood
            & ~flood_mask
            & (elev_data > floor_elev + 5)
            & (elev_data < floor_elev + 50)
            & (elev_data < 200)
            & np.isfinite(elev_data)
        )

    px      = int(np.sum(candidate))
    area_km2 = px * (RESOLUTION * 111) ** 2
    print(f"Candidate swale area : {px:,} pixels ≈ {area_km2:.0f} km²")
    print(f"(10 km buffer around {int(np.sum(flood_mask)):,} persistently flooded pixels)")
else:
    print("Run cells 2 (flood composite) and 3 (elevation) first.")

In [ ]:
candidate_path = DATA_DIR / "candidate_swale_zones.geojson"

if 'candidate' in dir():
    geoms = [
        shape(g)
        for g, v in rio_shapes(candidate.astype(np.uint8), transform=FREQ_TRANSFORM)
        if v == 1
    ]
    gdf_swale = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")

    # Add area attribute (rough, in km²)
    gdf_swale["area_km2"] = (
        gdf_swale.geometry.area * 111 * 111   # deg² → km²
    ).round(3)

    # Drop tiny fragments < 1 km²
    gdf_swale = gdf_swale[gdf_swale.area_km2 >= 1.0].reset_index(drop=True)
    gdf_swale["rank"] = gdf_swale.area_km2.rank(ascending=False).astype(int)
    gdf_swale.to_file(candidate_path, driver="GeoJSON")

    print(f"Exported {len(gdf_swale)} candidate zones → {candidate_path}")
    print(gdf_swale.nlargest(10, "area_km2")[["rank", "area_km2"]].to_string())

In [ ]:
m2 = leafmap.Map(center=CENTER, zoom=8, height="700px")
m2.add_basemap("CartoDB.DarkMatter")
m2.add_tile_layer(
    url=(
        "https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/"
        "SRTM_Color_Index/default/GoogleMapsCompatible_Level8/{z}/{y}/{x}.png"
    ),
    name="SRTM Color Index",
    opacity=0.55,
)

if freq_path.exists():
    m2.add_raster(
        str(freq_path), layer_name="Flood frequency",
        colormap="Blues", vmin=1, vmax=5, opacity=0.7, nodata=0,
    )

if candidate_path.exists():
    m2.add_geojson(
        str(candidate_path),
        layer_name="Candidate swale zones",
        style={"color": "#27ae60", "fillColor": "#2ecc71",
               "weight": 1, "fillOpacity": 0.50},
        hover_style={"fillOpacity": 0.8},
    )

m2.add_gdf(aoi_gdf, layer_name="AOI",
           style={"color": "#f1c40f", "fillOpacity": 0, "weight": 2})
m2

## 6 · GeoAgent natural language session

`geoagent.tools.nasa_opera` is QGIS-native (`display_raster` / `display_footprints`
require Qt). The tools that **do** work standalone in Jupyter are:
`search_opera_data`, `count_water_pixels`, `analyze_categorical_raster`.

We combine those with the akb knowledge-base tools so the agent can correlate
satellite flood data with the corpus.

Requires: `pip install 'akb[geoagent]'` and a valid Earthdata account.

In [ ]:
try:
    from geoagent.tools.nasa_opera import nasa_opera_tools
    from cli.geoagent_tools import make_agent, AKB_TOOLS

    opera_tools = nasa_opera_tools()   # 8 tool functions
    all_tools   = opera_tools + AKB_TOOLS

    m3     = leafmap.Map(center=CENTER, zoom=8)
    agent  = make_agent(map=m3, tools_override=all_tools)

    agent.run(
        "Search for OPERA DSWx-HLS surface water data over the Anambra and Enugu "
        "lowlands of Nigeria (bbox: 4.25, 4.88, 9.30, 7.73) for October–November 2022. "
        "Count the open-water pixels (class 1) per tile and identify which areas had "
        "the highest flood extent. Then search the knowledge base for any historical "
        "records of flooding, water management infrastructure, or climate patterns in "
        "this region, and synthesise a brief landscape summary linking the satellite "
        "observations to the historical record."
    )
    m3
except ImportError as e:
    print(f"GeoAgent not installed ({e}). Install with: pip install 'akb[geoagent]'")
    print("The analysis in cells 1–5 runs without GeoAgent.")

## 7 · Historical context from the akb corpus

Query the local knowledge base for what the ingested documents say about
flooding, water management, and climate in this region — connecting satellite
observations to the archive's historical and policy literature.

In [ ]:
from cli.geoagent_tools import akb_search_locations, akb_get_entity_network

results = akb_search_locations(
    "flood water management Niger Anambra Enugu Nigeria climate",
    top_k=12,
)
print(f"{len(results['features'])} geocoded locations from corpus search:\n")
for f in results['features'][:8]:
    p = f['properties']
    coords = f['geometry']['coordinates']
    print(f"  [{p['name']:30s}]  score={p['rrf_score']:.3f}  t={p['time_context'] or '—'}")
    print(f"    {p['excerpt'][:120]}…")
    print()

In [ ]:
# What does the corpus say specifically about Anambra?
network = akb_get_entity_network("Anambra")
print(f"{network['occurrence_count']} chunks mention Anambra\n")
for occ in network['occurrences'][:4]:
    print(f"[{occ['block_title']}]")
    print(f"  {occ['excerpt'][:180]}…")
    for typ, vals in occ['co_entities'].items():
        print(f"  {typ}: {', '.join(vals[:5])}")
    print()

In [ ]:
# Overlay corpus locations on the flood frequency map
m4 = leafmap.Map(center=CENTER, zoom=8, height="700px")
m4.add_basemap("CartoDB.DarkMatter")

if freq_path.exists():
    m4.add_raster(
        str(freq_path), layer_name="Flood frequency",
        colormap="Blues", vmin=1, vmax=5, opacity=0.70, nodata=0,
    )

# Corpus search results as points
m4.add_geojson(
    results,
    layer_name="akb corpus: flood / water locations",
    style={"color": "#f39c12", "fillColor": "#f39c12",
           "radius": 6, "fillOpacity": 0.85, "weight": 1},
    point_style={"radius": 6},
    info_mode="on_hover",
)
m4

## Export

Files written to `data/opera_dswx/`:

| File | Description |
|---|---|
| `flood_frequency.tif` | Uint8 raster, 0–5 seasons flooded |
| `srtm_nigeria_90m.tif` | Raw SRTM 90 m DEM |
| `srtm_nigeria_30m.tif` | SRTM resampled to freq grid |
| `candidate_swale_zones.geojson` | Candidate water-retention zones |

Load `flood_frequency.tif` directly in QGIS / Google Earth Pro (GeoTIFF),
or drag `candidate_swale_zones.geojson` into any GIS tool for field planning.